In [3]:
import duckdb 

In [4]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

IOException: IO Error: Cannot open file "C:\Users\erick\OneDrive\Documentos\WorkSpaces\myproject\scripts\dados_duckdb.db": O arquivo já está sendo usado por outro processo.

File is already open in 
C:\Users\erick\AppData\Local\Programs\Python\Python311\python.exe (PID 53512)

In [ ]:
df = con.execute("SELECT * FROM bronze_z0019").fetchdf()
df.head(10)

In [ ]:
df = con.execute("""SELECT * FROM (SELECT 
                       *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC)  
                           as row_num FROM bronze_z0019 WHERE data_ingestao >= '2026-10-14') where row_num = 1""").fetchdf()
df.head(10)

In [ ]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row_num'])
df_final = df_final.rename(columns={'NATBR': 'id',})
df_final = df_final.rename(columns={'MAKTX': 'nm_produto',})
df_final = df_final.rename(columns={'WERKS': 'id_categoria',})
df_final = df_final.rename(columns={'MAINS': 'id_fornecedor',})
df_final = df_final.rename(columns={'LABST': 'vl_preco',})
df_final.head(10)

In [ ]:
df2 = df_final
df2 = df2.astype({'id': 'int64', 'nm_produto': 'string', 'id_categoria': 'string', 'id_fornecedor': 'int64', 'vl_preco': 'float64'})
df2.dtypes

In [ ]:
con.execute("CREATE TABLE IF NOT EXISTS produtos (id BIGINT, nm_produto TEXT, id_categoria TEXT, id_fornecedor BIGINT, vl_preco REAL)")


In [ ]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [ ]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

In [ ]:
con.close()